# Step 1: Clean and Prepare Data

Kilian Lüders & Hannah Birkenkötter

Parses and segments the documents and creates clean data table for the analysis.

**Steps:**
1. Load Data
2. Preprocessing & Cleaning
3. Save Dataset

**Input:**
- `data/CR-UNSC_2024-05-19_ALL_CSV_FULL.csv` – csv file from [https://zenodo.org/records/11212056]
- `data/handcoding.json`

**Output:**
- `data/full_data.pkl` – full data on paragraph level (59685 rows x 3 columns); columns: `segmentClass`, `text`, `doc`

In [1]:
import os
import re
import json
import numpy as np
import pandas as pd

### 1. Load Data

In [2]:
# main data source

data = pd.read_csv("data/CR-UNSC_2024-05-19_ALL_CSV_FULL.csv")
data['doc'] = data['doc_id'].apply(lambda x: x.replace(".txt", "").replace("_EN", "").replace("_GOLD", ""))
data['date'] = pd.to_datetime(data.date)
data.shape

(2722, 83)

In [3]:
# HANDCODING
# Some documents have been manually corrected:
# OCR errors were fixed andthe document layout was standardized to follow a consistent schema.

with open("data/handcoding.json", "r") as f:
    handcodings = json.load(f)

for coding in handcodings:
    id = next(iter(coding))
    data.loc[data.doc == id, "text"] = coding[id]

In [4]:
# exclude resolution from 2024
# our analysis focuses on the 2721 resolutions in the dataset from 1946 to 2023

data = data[~data.doc_id.isin([
    "S_RES_2722_2024_EN.txt"
    ])]
data.shape

(2721, 83)

### 2. Preprocessing & Cleaing

The following functions are used to clean up the documents by removing layout elements, splitting text into paragraphs, and filtering out unnecessary labels.

In [5]:
pattern_main_start  = re.compile(r'\s+The Security Council,?\n') # start main poart of doc
pattern_main_end  = re.compile(r'(?:\.)\n+?\s*?(Annex|Adopted unanimously at the|Adopted at the|Procedures)') # end main poart of doc
pattern_page_break =  re.compile(r'\n+?[*\- ()/\.\d A-Z\n(?:English|Page)]+\n') # page breaks
pattern_split_op = re.compile(r'\n(?:\n| )+1\.') # split op and pp; takes "1." to split
pattern_split_op_heading = re.compile(r',\n\n([a-zA-Z\s,]+[a-zA-Z]+)\n(?:\n| )+1\.')
pattern_op_len_count = re.compile(r'[:;,]\n\s*(\d{1,3})[\.,]')
pattern_split_num = re.compile(r'[:;,]\n\n?(?: *|\n)?\d{1,3}[\.,]') # split paragraphs in op section
pattern_s_strip = re.compile(r"^\s+") # matches space offset
pattern_split_num_1 = re.compile(r' *?1\.') # matches numb first line of op section
pattern_ws_b = re.compile(r"^ +") # matches space offset
pattern_op_split_heading = re.compile(r'[:;]\n\n? *([\w \’’ô\,-]+)\n *\d{1,3}[\.,]') # matches headings in op section
pattern_split_pp = re.compile(r',\n(?: +|\n+)') # split paragraphs in pp section
pattern_spaces = re.compile(r"\s+") # matches all spaces to strip
pattern_line = re.compile(r"\n *[_-`]+\n")
pattern_reissued = re.compile(r'\n *\*? *Reissued for technical reasons.\n')


def clean_offset(line, offset):
    return line.replace(" ", "", offset)

def clean_offset(line, offset):
    count_sp = len(line) - len(line.lstrip())
    if count_sp >= offset:
        return line.replace(" ", "", offset)
    return line.replace(" ", "", count_sp)

def count_offset(line):
    ws_match = re.search(pattern_ws_b, line)
    if ws_match == None:
        return 0
    return len(ws_match.group())

def reset_spaces(page):
    if page == "":
        return page
    offset = min([count_offset(line) for line in page.split("\n") if len(line)>0])
    return "\n".join([clean_offset(line, offset) for line in page.split("\n")])

def reset_spaces(page):
    if page == "":
        # empty page
        return page
    offset = [count_offset(line) for line in page.split("\n") if len(line)>0]
    offset.sort()
    page = "\n".join([clean_offset(line, offset[0]) for line in page.split("\n")])
    if len(offset) == 1:
        # only one line
        return page
    if offset[0] != offset[1]:
        page = "\n".join([clean_offset(line, offset[1] - offset[0]) for line in page.split("\n")])
    return page

def clean_pp(pp_text):
    dec_data = list()
    for p in re.split(pattern_split_pp, pp_text):
        p_clean = re.sub(pattern_spaces, " ", p)
        dec_data.append(
            {'segmentClass': 'content',
             'text': p_clean.strip()
             })
    return dec_data

def get_op_count(raw_text):
    num_list = re.findall(pattern_op_len_count, raw_text)
    if len(num_list) > 0:
        return int(num_list[-1])
    else:
        return 1

def clean_op(op_text, dec = ""):
    op_count = 0
    dec_data = list()

    op_len = get_op_count(op_text)

    op_split_heading = re.split(pattern_op_split_heading, op_text)
    # iterative over heading splits in op
    for i, e in enumerate(op_split_heading):
        if (i + 1) % 2 == 0:
            # heading
            dec_data.append(
                {'segmentClass': 'heading',
                 'text': e})
        else:
            # operative text paragraph
            if i == 0:
                e = re.sub(pattern_split_num_1, "", e, count=1)
            for p in re.split(pattern_split_num, e):
                op_count += 1
                p_clean = re.sub(pattern_spaces, " ", p)
                dec_data.append(
                    {'segmentClass': 'content_num',
                     'text': p_clean.strip()})
    if op_len != op_count:
        raise ValueError("Op Count stimmt nicht: {}!".format(dec))
    return dec_data

def clean_doc(doc_name, raw_text, debug=False):
    # 1) clean text input
    raw_text = re.sub(pattern_line, "\n", raw_text)
    raw_text = raw_text.replace('\x0c', "\n\n")
    raw_text = re.sub(r'([;:,]) \d{1,3}\n\n', r'\1\n\n', raw_text) # footnotes
    raw_text = re.sub(pattern_reissued, "\n", raw_text)
    
    # 2) extract header
    match_start = re.search(pattern_main_start, raw_text)
    if match_start == None:
        print(doc_name)
        raise ValueError("No Start Main Document: {}".format(doc_name))
    head_text = raw_text[:match_start.end()]
    doc = [{'segmentClass': 'head',
        'text': re.sub(pattern_spaces, " ", head_text)
        }]
    raw_text = raw_text[match_start.end():]

    # 3) page breaks
    #clean page break
    raw_text = re.sub(pattern_page_break, '<PB>', raw_text)
    #clean space offset 
    raw_text = "\n".join([reset_spaces(page) for page in raw_text.split("<PB>")])
    
    # 4) document tail
    # cut out doc tail
    match_end = re.search(pattern_main_end, raw_text)
    if match_end == None:
        match_end_pos = len(raw_text)
    else:
        match_end_pos = match_end.start(1)
    tail_text = raw_text[match_end_pos:]
    raw_text = raw_text[:match_end_pos]

    if debug:
        return raw_text

    # 5) split op from pp & process pp/op
    # with headline before op
    split_op_heading = re.split(pattern_split_op_heading, raw_text)
    if len(split_op_heading) == 3:
        # found headline
        doc += clean_pp(split_op_heading[0])
        doc += [{'segmentClass': 'heading',
                'text': re.sub(pattern_spaces, " ", split_op_heading[1])
                }]
        # split swallows "1."
        op_text = "1. " + split_op_heading[2]
        doc += clean_op("1. " + split_op_heading[2], doc_name)
    else: 
        match_op = re.search(pattern_split_op, raw_text)
        if match_op == None:
            # no op at all
            doc += clean_pp(raw_text)
        else:
            # op-split found
            doc += clean_pp(raw_text[:match_op.start()+1])
            doc += clean_op(raw_text[match_op.start()+1:], doc_name)
            op_text = raw_text[match_op.start()+1:]
    if len(tail_text) > 0:
        # tail found
        doc += [{'segmentClass': 'tail',
                'text': re.sub(pattern_spaces, " ", tail_text)
                }]
    for para in doc:
        para['doc'] = doc_name
    return doc

In [6]:
full_doc = list()

for i,idx in enumerate(data.index):
    print("{} / {}".format(i+1,len(data)), end="\r")
    name = data['doc_id'][idx].replace(".txt", "").replace("_EN", "").replace("_GOLD", "")
    #print(name)
    text = data['text'][idx]
    #with open("test_data/" + name + "_raw.txt", "w") as text_file:
    #    text_file.write(text)
    doc = clean_doc(name,text)
    full_doc.append(pd.DataFrame(doc))

### 3. Save Dataset

In [7]:
full_data = pd.concat(full_doc)
full_data

,segmentClass,text,doc
0,head,MILITARY STAFF COMMITTEE 1 (1946) Resolution o...,S_RES_0001_1946
1,content,Therefore,S_RES_0001_1946
2,content_num,Requests the permanent members of the Security...,S_RES_0001_1946
3,content_num,Directs that the Chiefs of Staff or their repr...,S_RES_0001_1946
4,content_num,Directs the Military Staff Committee thereupon...,S_RES_0001_1946
...,...,...,...
11,content_num,Encourages member states and all other relevan...,S_RES_2721_2023
12,content_num,"Requests the Secretary-General, in consultatio...",S_RES_2721_2023
13,content_num,Welcomes the Secretary-General’s intention to ...,S_RES_2721_2023
14,content_num,Requests that the Secretary-General brief the ...,S_RES_2721_2023


In [8]:
# drop empty lines
full_data = full_data[full_data.text != ""]
print(full_data.shape)

(59685, 3)


In [9]:
full_data.to_pickle("data/full_data.pkl")